In [19]:
import pandas as pd
import statsmodels.api as sm
import numpy as np

n = 100

df = pd.DataFrame({
    "x1": np.random.normal(0, 1, n),
    "x2": np.random.normal(0, 1, n)
})

#Generate y 
df["y"] = (
    5
    + 2 * df["x1"]
    - 1.5 * df["x2"]
    + np.random.normal(0, 1, n)
)

#Add outliers
df.loc[10, "y"] += 20
df.loc[40, "y"] -= 25
df.loc[75, "y"] += 30

df.head()

,x1,x2,y
0,0.419366,-0.861170,6.214764
1,-0.227489,2.386431,1.640734
2,-0.187694,0.725776,2.226805
3,-1.627107,2.733582,-1.402162
4,1.306097,-0.234371,7.436249


In [20]:
X = sm.add_constant(df[["x1", "x2"]])
y = df["y"]

model = sm.OLS(y, X)
result = model.fit()
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.177
Model:                            OLS   Adj. R-squared:                  0.160
Method:                 Least Squares   F-statistic:                     10.43
Date:                Tue, 28 Jul 2026   Prob (F-statistic):           7.89e-05
Time:                        12:17:08   Log-Likelihood:                -290.68
No. Observations:                 100   AIC:                             587.4
Df Residuals:                      97   BIC:                             595.2
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          5.1257      0.454     11.303      0.000       4.226       6.026
x1             1.7358      0.435      3.992      0.000       0.873       2.599
x2            -1.1301      0.455     -2.486      0.015      -2.032      -0.228
==============================================================================
Omnibus:                       88.995   Durbin-Watson:                   2.084
Prob(Omnibus):                  0.000   Jarque-Bera (JB):             3601.657
Skew:                           2.158   Prob(JB):                         0.00
Kurtosis:                      32.082   Cond. No.                         1.17
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""

In [21]:
#Gather residuals and calculate standardized residuals
df['yhat'] = result.fittedvalues
df['e'] = result.resid

df['e_stdev'] = df['e'].std()
df['standardized_e'] = df['e']/df['e_stdev']

In [22]:
#Identify outliers
outlier_indices = df.index[
    df["standardized_e"].abs() > 3
]


print("Outlier observation indexes:")
print(outlier_indices.tolist())

Outlier observation indexes:
[10, 40, 75]


In [23]:
#Remove Outliers
df_original = df.copy() #Keep the original data
df = df.drop(index=outlier_indices)

In [24]:
len(df)

97

In [25]:
X = sm.add_constant(df[["x1", "x2"]])
y = df["y"]

model = sm.OLS(y, X)
result = model.fit()
result.summary()

<class 'statsmodels.iolib.summary.Summary'>
"""
                            OLS Regression Results                            
==============================================================================
Dep. Variable:                      y   R-squared:                       0.865
Model:                            OLS   Adj. R-squared:                  0.862
Method:                 Least Squares   F-statistic:                     300.1
Date:                Tue, 28 Jul 2026   Prob (F-statistic):           1.55e-41
Time:                        12:17:21   Log-Likelihood:                -143.18
No. Observations:                  97   AIC:                             292.4
Df Residuals:                      94   BIC:                             300.1
Df Model:                           2                                         
Covariance Type:            nonrobust                                         
==============================================================================
                 coef    std err          t      P>|t|      [0.025      0.975]
------------------------------------------------------------------------------
const          4.9207      0.111     44.499      0.000       4.701       5.140
x1             2.1726      0.105     20.692      0.000       1.964       2.381
x2            -1.5962      0.111    -14.398      0.000      -1.816      -1.376
==============================================================================
Omnibus:                        0.767   Durbin-Watson:                   2.125
Prob(Omnibus):                  0.681   Jarque-Bera (JB):                0.885
Skew:                           0.148   Prob(JB):                        0.642
Kurtosis:                       2.637   Cond. No.                         1.20
==============================================================================

Notes:
[1] Standard Errors assume that the covariance matrix of the errors is correctly specified.
"""